# U-Net Glomeruli Segmentation — Binary Segmentation Training Pipeline

This notebook implements a complete PyTorch U-Net training system for binary semantic segmentation of glomeruli from renal biopsy whole-slide images.

## Two-Model Strategy

This pipeline uses a **two-stage approach** for glomeruli classification:

1. **Model 1 (this notebook)**: Binary segmentation to detect glomerulus locations
   - Input: Z-score normalized RGB tiles from WSI
   - Output: Pixel-level binary masks (background vs glomerulus)
   - Classes: 0 (background), 1 (glomerulus — fused from original classes 1-4)
   - Purpose: Localizes all glomeruli regardless of subtype

2. **Model 2 (future)**: Classifier on detected patches to assign classes 1-4
   - Input: Image patches extracted from detected glomerulus regions (Model 1)
   - Output: Per-glomerulus classification (classes 1-4: FGS, MPGN, FSGS, other)
   - Purpose: Fine-grained subtype classification for clinical diagnosis

## Notebook Contents

1. **Data Loading** — GlomeruliDataset with grouped split and online augmentation (train only)
2. **Loss Functions** — BCEWithLogits + Dice for binary segmentation
3. **U-Net Architecture** — Encoder-Decoder with Skip Connections
4. **Binary Segmentation Metrics** — Accuracy, Precision, Recall, F1, ROC-AUC, Dice
5. **Training Pipeline** — Full training loop with validation and checkpointing

## Pipeline Overview

WSI TIFF + GeoJSON annotations → Tiled images + masks (classes 0-4) →
Reinhard color normalization → Z-score standardization →
Train/Val/Test split (grouped by biopsy, 70/15/15) →
Online augmentation (train only) → U-Net training → Model evaluation (Binary Metrics)

See WORKFLOW.md for end-to-end pipeline documentation.

In [1]:
# Standard library
import os
import json
import time
from pathlib import Path
from datetime import datetime
from typing import Tuple
import random

# Scientific computing
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
import albumentations as A

# Data loading
import cv2
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, WeightedRandomSampler
import warnings

# Progress/model library imports
from tqdm.auto import tqdm
import segmentation_models_pytorch as smp
from PIL import Image, ImageFile

# Allow PIL to load truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True
# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# --- CUDA performance tuning (T4 Tensor Cores) ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True          # auto-tune conv algorithms
    torch.backends.cuda.matmul.allow_tf32 = True   # TF32 for matmul (T4 supports it)
    torch.backends.cudnn.allow_tf32 = True         # TF32 for cudnn convolutions
    print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}")
    print(f"CUDA matmul TF32: {torch.backends.cuda.matmul.allow_tf32}")
    print(f"cudnn TF32: {torch.backends.cudnn.allow_tf32}")
    print(f"PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True")


# --- Preprocessing Transforms (Reinhard Normalization + Z-score) ---

class ReinhardNormalize:
    """Reinhard stain normalization in LAB color space for consistency across slides."""

    def __init__(self, target_stats: dict):
        self.target = target_stats

    @staticmethod
    def _get_tissue_mask(img_bgr: np.ndarray) -> np.ndarray:
        """Isolate tissue pixels from background and artifacts via luminance and saturation thresholds."""
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        mask = (lab[:, :, 0] < 230) & (hsv[:, :, 1] > 10)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
        mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        return mask.astype(bool)

    @staticmethod
    def compute_template_stats(image_paths: list, n_samples: int = 200) -> dict:
        """Compute median LAB statistics from tissue pixels to define normalization target."""
        paths_list = list(image_paths)[:n_samples]
        sample = random.sample(paths_list, min(n_samples, len(paths_list)))
        all_stats = []
        
        for p in sample:
            img = cv2.imread(str(p))
            if img is None:
                continue
            tissue = ReinhardNormalize._get_tissue_mask(img)
            if tissue.sum() < 100:
                continue
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
            stats = ([lab[..., c][tissue].mean() for c in range(3)] +
                     [lab[..., c][tissue].std()  for c in range(3)])
            all_stats.append(stats)
        
        if not all_stats:
            return {'mean_L': 50, 'mean_a': 128, 'mean_b': 128,
                    'std_L': 10, 'std_a': 10, 'std_b': 10}
        
        arr = np.array(all_stats)
        keys = ['mean_L', 'mean_a', 'mean_b', 'std_L', 'std_a', 'std_b']
        return {k: float(np.median(arr[:, i])) for i, k in enumerate(keys)}

    def __call__(self, img_bgr: np.ndarray) -> np.ndarray:
        """Normalize image to match template statistics, preserving background pixels."""
        tissue = self._get_tissue_mask(img_bgr)
        if tissue.sum() < 100:
            return img_bgr
        
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        
        src = {c: (lab[..., i][tissue].mean(), lab[..., i][tissue].std())
               for i, c in enumerate(['L', 'a', 'b'])}
        
        result = lab.copy()
        for i, c in enumerate(['L', 'a', 'b']):
            m, s = src[c]
            result[..., i] = ((lab[..., i] - m) *
                              (self.target[f'std_{c}'] / (s + 1e-5)) +
                              self.target[f'mean_{c}'])
        
        result[~tissue] = lab[~tissue]
        result = np.clip(result, 0, 255).astype(np.uint8)
        return cv2.cvtColor(result, cv2.COLOR_LAB2BGR)


def compute_channel_stats(image_paths: list, n_samples: int = 200) -> Tuple[list, list]:
    """Compute weighted per-channel RGB stats from tissue pixels for Z-score normalization."""
    paths_list = list(image_paths)[:n_samples]
    sample = random.sample(paths_list, min(n_samples, len(paths_list)))
    
    tile_means, tile_vars, tile_counts = [], [], []
    skipped_count = 0
    for p in sample:
        img_bgr = cv2.imread(str(p))
        if img_bgr is None:
            skipped_count += 1
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        tissue = ReinhardNormalize._get_tissue_mask(img_bgr)
        if tissue.sum() < 100:
            skipped_count += 1
            continue
        
        pixels = img_rgb[tissue]
        tile_means.append(pixels.mean(0))
        tile_vars.append(pixels.var(0))
        tile_counts.append(tissue.sum())
    
    if not tile_means:
        print(f"  ⚠️  WARNING: No valid tissue pixels found in {len(sample)} samples!")
        print(f"      {skipped_count} samples were skipped (tissue.sum() < 100 or read failed)")
        print(f"      Falling back to default values [0.5, 0.5, 0.5] and [0.2, 0.2, 0.2]")
        print(f"      This will make Z-score normalization INEFFECTIVE.")
        return [0.5, 0.5, 0.5], [0.2, 0.2, 0.2]
    
    print(f"  ✓ Computed stats from {len(tile_means)} valid samples (skipped {skipped_count})")
    
    means_arr = np.array(tile_means)
    vars_arr = np.array(tile_vars)
    counts_arr = np.array(tile_counts, dtype=np.float64)
    w = counts_arr / counts_arr.sum()
    
    mean = (means_arr * w[:, None]).sum(0)
    var = ((vars_arr + (means_arr - mean) ** 2) * w[:, None]).sum(0)
    return mean.tolist(), var.tolist()

c:\Users\proyecto_final\Documents\Proyecto_Final_Glomerulos\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: Tesla T4
VRAM: 17.00 GB
cudnn.benchmark: True
CUDA matmul TF32: True
cudnn TF32: True
PYTORCH_CUDA_ALLOC_CONF: expandable_segments:True


In [2]:
def _collect_image_tiles(images_dir: str) -> list:
    """Collect only input tiles from */images/*.png, never generated masks."""
    images_dir = Path(images_dir)
    image_paths = sorted(images_dir.glob('*/images/*.png'))

    # Fallback for flat/custom datasets: include PNGs except anything inside a masks folder
    # or files already named *_mask.png.
    if not image_paths:
        image_paths = sorted(
            p for p in images_dir.rglob('*.png')
            if 'masks' not in p.relative_to(images_dir).parts
            and not p.stem.endswith('_mask')
        )

    return image_paths


def _slide_name_from_image_path(image_path, images_dir) -> str:
    """Infer the biopsy/slide name from a tile path under <root>/<slide>/images/*.png."""
    image_path = Path(image_path)
    images_dir = Path(images_dir)
    try:
        rel = image_path.relative_to(images_dir)
        if rel.parts:
            return rel.parts[0]
    except ValueError:
        pass

    if image_path.parent.name == 'images' and image_path.parent.parent.name:
        return image_path.parent.parent.name
    return image_path.parent.name or image_path.stem


def split_biopsias(
    images_dir: str,
    train_size: float = 0.70,
    val_size: float = 0.15,
    seed: int = 42,
) -> Tuple[list, list, list, dict]:
    """Groups biopsias into train/val/test to prevent data leakage at slide level."""
    images_dir = Path(images_dir)

    all_images = _collect_image_tiles(images_dir)

    if not all_images:
        raise ValueError(f"No PNG image tiles found in {images_dir}. Expected files under */images/*.png")

    biopsias_dict = {}
    for img_path in all_images:
        relative = img_path.relative_to(images_dir)
        biopsia = relative.parts[0]
        if biopsia not in biopsias_dict:
            biopsias_dict[biopsia] = []
        biopsias_dict[biopsia].append(img_path)

    biopsias_list = list(biopsias_dict.keys())

    test_size = 1.0 - train_size - val_size
    assert test_size >= 0, "train_size + val_size must be <= 1.0"

    if test_size > 0:
        train_val_biopsias, test_biopsias = train_test_split(
            biopsias_list,
            test_size=test_size,
            random_state=seed,
        )
    else:
        train_val_biopsias = biopsias_list
        test_biopsias = []

    if val_size > 0:
        val_fraction = val_size / (train_size + val_size)
        train_biopsias, val_biopsias = train_test_split(
            train_val_biopsias,
            test_size=val_fraction,
            random_state=seed + 1,
        )
    else:
        train_biopsias = train_val_biopsias
        val_biopsias = []

    return train_biopsias, val_biopsias, test_biopsias, biopsias_dict


# ============================================================================
# Grayscale pixel values that correspond to glomerulus classes (any > 0 in practice)
# 64=No_Proliferativo, 128=Proliferativo, 192=Esclerosado, 255=Excluido/Excluyente
# Excluido (255) is INTENTIONALLY mapped to Glomerulus class 1 for binary segmentation
# ============================================================================
class GlomeruliDataset(Dataset):
    """Loads paired image-mask glomeruli tiles with online preprocessing and augmentation."""

    def __init__(
        self,
        images_dir: str,
        masks_dir: str = None,
        split: str = 'train',
        biopsias: list = None,
        biopsias_dict: dict = None,
        reinhard_norm=None,
        channel_means: list = None,
        channel_stds: list = None,
        train_size: float = 0.70,
        val_size: float = 0.15,
        seed: int = 42,
        transforms=None,
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir) if masks_dir is not None else Path(images_dir)
        self.split = split
        self.transforms = transforms
        self.reinhard_norm = reinhard_norm
        self.channel_means = channel_means if channel_means is not None else [0.5, 0.5, 0.5]
        self.channel_stds = channel_stds if channel_stds is not None else [0.2, 0.2, 0.2]

        assert split in {'train', 'val', 'test'}, f"Invalid split: {split}"
        assert self.images_dir.exists(), f"Images dir not found: {self.images_dir}"
        assert self.masks_dir.exists(), f"Masks dir not found: {self.masks_dir}"

        if biopsias is not None and biopsias_dict is not None:
            selected_biopsias = biopsias
            self.biopsias_dict = biopsias_dict
        else:
            test_size = 1.0 - train_size - val_size
            assert test_size >= 0, "train_size + val_size must be <= 1.0"

            all_images = _collect_image_tiles(self.images_dir)
            if not all_images:
                raise ValueError(f"No PNG image tiles found in {self.images_dir}. Expected files under */images/*.png")

            biopsias_dict = {}
            for img_path in all_images:
                relative = img_path.relative_to(self.images_dir)
                biopsia = relative.parts[0]
                if biopsia not in biopsias_dict:
                    biopsias_dict[biopsia] = []
                biopsias_dict[biopsia].append(img_path)

            self.biopsias_dict = biopsias_dict
            biopsias_list = list(biopsias_dict.keys())

            if test_size > 0:
                train_val_biopsias, test_biopsias = train_test_split(
                    biopsias_list,
                    test_size=test_size,
                    random_state=seed,
                )
            else:
                train_val_biopsias = biopsias_list
                test_biopsias = []

            if val_size > 0:
                val_fraction = val_size / (train_size + val_size)
                train_biopsias, val_biopsias = train_test_split(
                    train_val_biopsias,
                    test_size=val_fraction,
                    random_state=seed + 1,
                )
            else:
                train_biopsias = train_val_biopsias
                val_biopsias = []

            if split == 'train':
                selected_biopsias = train_biopsias
            elif split == 'val':
                selected_biopsias = val_biopsias
            else:
                selected_biopsias = test_biopsias

        self.image_paths = []
        for biopsia in selected_biopsias:
            self.image_paths.extend(self.biopsias_dict[biopsia])

        self.image_paths = sorted(self.image_paths)

        paired = []
        missing = []
        for img_path in self.image_paths:
            mask_path = self._get_mask_path(img_path)
            if mask_path.exists():
                paired.append((img_path, mask_path))
            else:
                missing.append((img_path, mask_path))

        if missing:
            warnings.warn(
                f"Found {len(missing)} images without corresponding masks. "
                f"These will be skipped. First few: {missing[:3]}"
            )

        if not paired:
            raise ValueError("No valid image-mask pairs found after checking.")

        self.image_paths, self.mask_paths = zip(*paired)
        self.image_paths = list(self.image_paths)
        self.mask_paths = list(self.mask_paths)
        
        # Caches used by the train sampler/audits
        self._positive_flags = None
        self._tile_metadata = None
        self._annotation_tiles_by_key = None

    def _get_mask_path(self, image_path: Path) -> Path:
        """Convert */images/<tile>.png to */masks/<tile>_mask.png."""
        rel = image_path.relative_to(self.images_dir)
        parts = list(rel.parts)

        if len(parts) < 3 or parts[1] != 'images':
            raise ValueError(
                f"Unexpected image path layout: {image_path}. "
                "Expected <root>/<biopsia>/images/<tile>.png"
            )

        parts[1] = 'masks'
        stem = Path(parts[-1]).stem
        parts[-1] = f"{stem}_mask.png"
        return self.masks_dir / Path(*parts)

    def get_positive_flags(self) -> list:
        """Return list of bools: True if tile mask contains at least one glomerulus pixel.
        
        Used by WeightedRandomSampler. Reads mask files once at dataset init time.
        Masks are small enough (1024x1024 uint8 = 1MB) that this is feasible.
        Caches result to avoid double scan.
        """
        if self._positive_flags is not None:
            return self._positive_flags
        
        flags = []
        for mask_path in self.mask_paths:
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                flags.append(False)
            else:
                flags.append(bool(np.any(mask > 0)))
        self._positive_flags = flags
        return flags

    def _load_annotation_tiles_by_key(self) -> dict:
        """Index per-slide annotations.json entries by (slide_folder, tile image path).

        annotations.json is generated by the tiling step and records which
        glomeruli appear in each tile, their primary/secondary role, and their
        coverage percentage. The sampler uses this to avoid over-weighting the
        same glomerulus when it appears in multiple overlapping tiles.
        """
        if self._annotation_tiles_by_key is not None:
            return self._annotation_tiles_by_key

        tiles_by_key = {}
        for ann_path in sorted(self.images_dir.glob('*/annotations.json')):
            slide_folder = ann_path.parent.name
            try:
                with ann_path.open('r', encoding='utf-8') as f:
                    annotations = json.load(f)
            except Exception as exc:
                warnings.warn(f"Could not read annotations metadata from {ann_path}: {exc}")
                continue

            slide_name = annotations.get('slide') or slide_folder
            for tile in annotations.get('tiles', []):
                image_rel = tile.get('image')
                if not image_rel:
                    continue

                image_rel = str(Path(image_rel).as_posix())
                meta = dict(tile)
                meta['slide'] = slide_name
                meta['slide_folder'] = slide_folder
                meta['annotations_path'] = str(ann_path)

                # Support both the folder name and the slide name in case they differ.
                tiles_by_key[(slide_folder, image_rel)] = meta
                tiles_by_key[(slide_name, image_rel)] = meta

        self._annotation_tiles_by_key = tiles_by_key
        return tiles_by_key

    def get_tile_metadata(self, idx: int) -> dict:
        """Return annotations.json metadata for a dataset tile, or {} if unavailable."""
        if self._tile_metadata is None:
            tiles_by_key = self._load_annotation_tiles_by_key()
            metadata = []
            for image_path in self.image_paths:
                try:
                    rel = image_path.relative_to(self.images_dir)
                    slide_folder = rel.parts[0]
                    image_rel = Path(*rel.parts[1:]).as_posix()
                except Exception:
                    metadata.append({})
                    continue

                metadata.append(tiles_by_key.get((slide_folder, image_rel), {}))

            self._tile_metadata = metadata

        return self._tile_metadata[idx] if 0 <= idx < len(self._tile_metadata) else {}

    def get_sampling_weights(
        self,
        secondary_factor: float = 0.35,
        duplicate_aware: bool = True,
    ) -> Tuple[torch.Tensor, dict]:
        """Compute train-sampling weights with optional duplicate-aware glomerulus balancing.

        The fallback/simple mode balances positive vs negative tiles uniformly.
        The duplicate-aware mode uses annotations.json so each unique glomerulus
        contributes approximately one unit of positive sampling mass, distributed
        over all train tiles where it appears according to coverage_pct and role.
        """
        positive_flags = self.get_positive_flags()
        n_positive = int(sum(positive_flags))
        n_negative = int(len(positive_flags) - n_positive)

        report = {
            'mode': 'simple',
            'n_positive': n_positive,
            'n_negative': n_negative,
            'num_unique_glomeruli': 0,
            'primary_links': 0,
            'secondary_links': 0,
            'unmatched_positive_tiles': 0,
        }

        if n_positive == 0 or n_negative == 0:
            return None, report

        def simple_weights(mode: str = 'simple'):
            report['mode'] = mode
            weight_pos = 1.0 / n_positive
            weight_neg = 1.0 / n_negative
            return torch.tensor(
                [weight_pos if flag else weight_neg for flag in positive_flags],
                dtype=torch.float32,
            ), report

        if not duplicate_aware:
            return simple_weights('simple')

        glomerulus_entries = {}
        for idx, is_positive in enumerate(positive_flags):
            if not is_positive:
                continue

            meta = self.get_tile_metadata(idx)
            glomeruli = meta.get('glomeruli') if meta else None
            if not glomeruli:
                continue

            slide = meta.get('slide') or meta.get('slide_folder') or _slide_name_from_image_path(
                self.image_paths[idx], self.images_dir
            )
            for glom in glomeruli:
                glom_id = glom.get('id')
                if glom_id is None:
                    continue

                role = str(glom.get('role', 'primary')).lower()
                try:
                    coverage = max(float(glom.get('coverage_pct', 0.0)), 0.0) / 100.0
                except (TypeError, ValueError):
                    coverage = 0.0

                role_factor = secondary_factor if role == 'secondary' else 1.0
                raw_weight = max(coverage, 1e-6) * role_factor
                key = (str(slide), str(glom_id))
                glomerulus_entries.setdefault(key, []).append((idx, raw_weight, role))

                if role == 'secondary':
                    report['secondary_links'] += 1
                else:
                    report['primary_links'] += 1

        if not glomerulus_entries:
            warnings.warn(
                "Duplicate-aware sampler requested, but no usable glomerulus metadata was found. "
                "Falling back to simple positive/negative balancing."
            )
            return simple_weights('fallback-simple-no-annotations')

        positive_mass = np.zeros(len(positive_flags), dtype=np.float64)
        for entries in glomerulus_entries.values():
            total = sum(raw for _, raw, _ in entries)
            if total <= 0:
                continue
            for idx, raw, _ in entries:
                positive_mass[idx] += raw / total

        unmatched_positive_tiles = 0
        for idx, is_positive in enumerate(positive_flags):
            if is_positive and positive_mass[idx] <= 0:
                # Keep mask-positive tiles with missing/partial metadata trainable.
                positive_mass[idx] = 1.0
                unmatched_positive_tiles += 1

        total_positive_mass = float(positive_mass.sum())
        if total_positive_mass <= 0:
            warnings.warn(
                "Duplicate-aware sampler produced zero positive mass. "
                "Falling back to simple positive/negative balancing."
            )
            return simple_weights('fallback-simple-zero-positive-mass')

        weights = np.zeros(len(positive_flags), dtype=np.float64)
        for idx, is_positive in enumerate(positive_flags):
            if is_positive:
                weights[idx] = positive_mass[idx] / total_positive_mass
            else:
                weights[idx] = 1.0 / n_negative

        report.update({
            'mode': 'duplicate-aware',
            'num_unique_glomeruli': len(glomerulus_entries),
            'unmatched_positive_tiles': unmatched_positive_tiles,
            'positive_weight_sum': float(weights[np.array(positive_flags, dtype=bool)].sum()),
            'negative_weight_sum': float(weights[~np.array(positive_flags, dtype=bool)].sum()),
            'secondary_factor': secondary_factor,
        })
        return torch.tensor(weights, dtype=torch.float32), report

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Load BGR tile -> Reinhard normalization -> BGR-to-RGB -> augment -> Z-score -> tensors."""
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        # Load image with PIL for truncated image tolerance
        try:
            img_pil = Image.open(str(img_path)).convert('RGB')
            img_rgb_raw = np.array(img_pil)
            img_bgr = cv2.cvtColor(img_rgb_raw, cv2.COLOR_RGB2BGR)
        except Exception as e:
            raise RuntimeError(f"Failed to load image {img_path}: {e}")

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Failed to load mask: {mask_path}")

        # Binarize: 0 = background, any >0 value = glomerulus
        mask_binary = (mask > 0).astype(np.uint8)

        # 1. Apply Reinhard normalization on original BGR
        if self.reinhard_norm is not None:
            img_bgr = self.reinhard_norm(img_bgr)

        # 2. Convert to RGB once
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # 3. Apply augmentations (including ColorJitter) on normalized RGB
        if self.transforms is not None and self.split == 'train':
            augmented = self.transforms(image=img_rgb, mask=mask_binary)
            img_rgb = augmented['image']
            mask_binary = augmented['mask']

        # 4. Z-score normalization
        img_rgb = img_rgb.astype(np.float32) / 255.0
        if self.channel_means is not None and self.channel_stds is not None:
            for c in range(3):
                img_rgb[..., c] = (img_rgb[..., c] - self.channel_means[c]) / (self.channel_stds[c] + 1e-6)

        img_tensor = torch.from_numpy(np.transpose(img_rgb, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(mask_binary.astype(np.int64)).long()

        return img_tensor, mask_tensor

In [3]:
def create_dataloaders(
    images_dir: str,
    masks_dir: str = None,
    batch_size: int = 4,
    num_workers: int = 4,
    seed: int = 42,
    train_transforms=None,
    val_transforms=None,
    reinhard_norm=None,
    channel_means: list = None,
    channel_stds: list = None,
):
    """Create train/val/test DataLoaders with pre-computed grouped splits by biopsia and positive/negative balancing."""
    train_biopsias, val_biopsias, test_biopsias, biopsias_dict = split_biopsias(
        images_dir=images_dir,
        train_size=0.70,
        val_size=0.15,
        seed=seed,
    )

    train_ds = GlomeruliDataset(
        images_dir,
        masks_dir,
        split='train',
        biopsias=train_biopsias,
        biopsias_dict=biopsias_dict,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
        transforms=train_transforms,
    )

    val_ds = GlomeruliDataset(
        images_dir,
        masks_dir,
        split='val',
        biopsias=val_biopsias,
        biopsias_dict=biopsias_dict,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
        transforms=val_transforms,
    )

    test_ds = GlomeruliDataset(
        images_dir,
        masks_dir,
        split='test',
        biopsias=test_biopsias,
        biopsias_dict=biopsias_dict,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
        transforms=None,
    )

    # ========== WeightedRandomSampler: positive/negative balance + duplicate-aware positives ==========
    print("Computing duplicate-aware sampling weights for WeightedRandomSampler...")
    positive_flags = train_ds.get_positive_flags()
    n_positive = sum(positive_flags)
    n_negative = len(positive_flags) - n_positive
    print(f"  Positive tiles (contain glomerulus): {n_positive:,}")
    print(f"  Negative tiles (background only):    {n_negative:,}")

    sample_weights, sampler_report = train_ds.get_sampling_weights(
        secondary_factor=0.35,
        duplicate_aware=True,
    )

    if sample_weights is None:
        warnings.warn(
            f"Cannot balance sampler: positives={n_positive}, negatives={n_negative}. "
            "Falling back to normal shuffle."
        )
        sampler = None
        shuffle_train = True
    else:
        num_samples = min(len(positive_flags), 2 * n_positive)
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=num_samples,
            replacement=True,
        )
        shuffle_train = False
        print(f"  Sampler mode: {sampler_report.get('mode')}")
        print(f"  Unique glomeruli in train metadata: {sampler_report.get('num_unique_glomeruli', 0):,}")
        print(f"  Primary tile links:   {sampler_report.get('primary_links', 0):,}")
        print(f"  Secondary tile links: {sampler_report.get('secondary_links', 0):,}")
        if sampler_report.get('unmatched_positive_tiles', 0):
            print(f"  Positive tiles without annotation metadata: {sampler_report['unmatched_positive_tiles']:,}")
        if 'positive_weight_sum' in sampler_report and 'negative_weight_sum' in sampler_report:
            print(
                f"  Weight mass: pos={sampler_report['positive_weight_sum']:.3f}, "
                f"neg={sampler_report['negative_weight_sum']:.3f}"
            )
        print(f"  WeightedRandomSampler: {num_samples:,} samples/epoch (~1:1 pos:neg ratio)")

    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=shuffle_train,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
        prefetch_factor=3 if num_workers > 0 else None,
    )

    val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2 if num_workers > 0 else None,
    )

    test_loader = torch.utils.data.DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2 if num_workers > 0 else None,
    )

    return train_loader, val_loader, test_loader

In [4]:
# Quick test of dataset loading
if Path('Salidas/Tiles_UNet').exists():
    ds_test = GlomeruliDataset('Salidas/Tiles_UNet', split='train')
    print(f"Dataset loaded: {len(ds_test)} tiles")
    print(f"First image path: {ds_test.image_paths[0]}")
    print(f"First mask path:  {ds_test.mask_paths[0]}")

    img, mask = ds_test[0]
    print(f"Image shape: {img.shape}, dtype: {img.dtype}")
    print(f"Mask shape: {mask.shape}, dtype: {mask.dtype}")
    print(f"Mask unique classes (binary): {torch.unique(mask).tolist()}")
    print(f"Class distribution: 0 (background)={torch.sum(mask == 0).item()}, 1 (glomerulus)={torch.sum(mask == 1).item()}")

    positive_idx = next((i for i, p in enumerate(ds_test.mask_paths) if cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).max() > 0), None)
    if positive_idx is not None:
        _, positive_mask = ds_test[positive_idx]
        print(f"Positive mask sanity check at idx={positive_idx}: classes={torch.unique(positive_mask).tolist()}, "
              f"glomerulus_pixels={torch.sum(positive_mask == 1).item()}")
    else:
        warnings.warn("No positive masks found in this split. Check tiling annotations or split selection.")
else:
    print("Dataset directory not found. Run preprocessing pipeline first.")
    print("Required: tiling_unet.py -> normalizacion.py")



Dataset loaded: 10712 tiles
First image path: Salidas\Tiles_UNet\18-139\images\18-139_centered_g0002.png
First mask path:  Salidas\Tiles_UNet\18-139\masks\18-139_centered_g0002_mask.png
Image shape: torch.Size([3, 1024, 1024]), dtype: torch.float32
Mask shape: torch.Size([1024, 1024]), dtype: torch.int64
Mask unique classes (binary): [0, 1]
Class distribution: 0 (background)=769779, 1 (glomerulus)=278797
Positive mask sanity check at idx=0: classes=[0, 1], glomerulus_pixels=278797


# Visual Data Exploration — Glomeruli Samples & Masks

In this section, we visualize the raw data: tiles from biopsies with their corresponding binary masks. 
The masks show glomerulus regions (red overlay) on the histological image.

In [ ]:
def visualize_tiles_and_masks(images_dir, n_tiles=6, figsize=(20, 8)):
    """
    Visualize random positive tiles with their binary masks overlaid in red.
    
    Args:
        images_dir: Root directory with tile data (Salidas/Tiles_UNet)
        n_tiles: Number of tiles to show
        figsize: Figure size
    """
    from pathlib import Path
    import matplotlib.pyplot as plt
    import cv2
    from PIL import Image
    
    # Get dataset to find positive tiles
    ds = GlomeruliDataset(images_dir, split='train', reinhard_norm=None, 
                          channel_means=None, channel_stds=None)
    positive_flags = ds.get_positive_flags()
    positive_indices = [i for i, flag in enumerate(positive_flags) if flag]
    
    if len(positive_indices) == 0:
        print("No positive tiles found")
        return
    
    # Sample n_tiles random positives
    import random
    random.seed(42)
    selected_indices = random.sample(positive_indices, min(n_tiles, len(positive_indices)))
    
    fig, axes = plt.subplots(2, len(selected_indices), figsize=figsize)
    if len(selected_indices) == 1:
        axes = axes.reshape(2, 1)
    
    for col, idx in enumerate(selected_indices):
        image_path = ds.image_paths[idx]
        mask_path = ds._get_mask_path(image_path)
        
        # Load image
        img_rgb = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
        
        # Load mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask_binary = (mask > 0).astype(np.uint8)
        
        # Row 0: Original image
        axes[0, col].imshow(img_rgb)
        axes[0, col].set_title(f"Tile {idx}\n{Path(image_path).parent.parent.name}", fontsize=10)
        axes[0, col].axis('off')
        
        # Row 1: Overlay mask on image
        overlay = img_rgb.copy().astype(float)
        overlay[mask_binary == 1] = [255, 0, 0]
        overlay = overlay.astype(np.uint8)
        
        # Blend
        alpha = 0.4
        blended = cv2.addWeighted(img_rgb, 1-alpha, overlay, alpha, 0)
        
        axes[1, col].imshow(blended)
        axes[1, col].set_title(f"Mask Overlay", fontsize=10)
        axes[1, col].axis('off')
    
    plt.suptitle("Sample Positive Tiles with Binary Masks", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
if Path('Salidas/Tiles_UNet').exists():
    visualize_tiles_and_masks('Salidas/Tiles_UNet', n_tiles=6, figsize=(20, 8))

In [5]:
class BinaryDiceLoss(nn.Module):
    """Sørensen-Dice coefficient for 1-channel binary logits (sigmoid output)."""

    def __init__(self, smooth: float = 1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits: [B, 1, H, W] raw logits from smp model
            targets: [B, H, W] int64 binary labels {0, 1}
        """
        probs = torch.sigmoid(logits).squeeze(1)           # [B, H, W]
        targets_f = targets.float()                        # [B, H, W]
        
        intersection = (probs * targets_f).sum(dim=(1, 2))
        cardinality   = probs.sum(dim=(1, 2)) + targets_f.sum(dim=(1, 2))
        
        dice_per_sample = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice_per_sample.mean()

In [6]:
class BinaryCombinedLoss(nn.Module):
    """Focal + Dice loss for binary segmentation, ignoring damaged pixels."""

    def __init__(
        self,
        weight_focal: float = 0.5,
        weight_dice: float = 0.5,
        smooth: float = 1e-6,
    ):
        super().__init__()
        self.weight_focal = weight_focal
        self.weight_dice = weight_dice
        self.focal_loss = smp.losses.FocalLoss(mode="binary")
        self.dice_loss = smp.losses.DiceLoss(mode="binary", smooth=smooth)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits:  [B, 1, H, W] raw logits from smp model
            targets: [B, H, W] int64 binary labels
        """
        targets_f = targets.float().unsqueeze(1)           # [B, 1, H, W]
        
        # Create boolean mask for valid pixels (not damaged)
        valid_mask = (targets >= 0) & (targets <= 1)       # [B, H, W]
        
        # For focal loss: use boolean indexing to select only valid pixels
        logits_valid = logits.squeeze(1)[valid_mask]       # [N_valid]
        targets_valid = targets_f.squeeze(1)[valid_mask]   # [N_valid]
        focal = F.binary_cross_entropy_with_logits(logits_valid, targets_valid, reduction='mean')
        
        # For Dice loss: treat invalid pixels as background (0) in the full image
        targets_valid_img = targets_f.clone()
        targets_valid_img[~valid_mask.unsqueeze(1)] = 0
        dice = self.dice_loss(logits, targets_valid_img)
        
        return self.weight_focal * focal + self.weight_dice * dice


In [7]:
# Smoke test for binary loss functions (1-channel logits)
B, H, W = 2, 256, 256
logits = torch.randn(B, 1, H, W, device=device, requires_grad=True)
targets = torch.randint(0, 2, (B, H, W), device=device)

dice_loss = BinaryDiceLoss()
dice = dice_loss(logits, targets)
print(f"Binary Dice Loss: {dice.item():.4f}")

combined = BinaryCombinedLoss()
loss = combined(logits, targets)
print(f"Binary Combined Loss: {loss.item():.4f}")

loss.backward()
print("✓ Backward pass OK")

Binary Dice Loss: 0.5007
Binary Combined Loss: 0.4242
✓ Backward pass OK


## Part 3: Model — segmentation_models_pytorch U-Net with ResNet-34 backbone

The executable model definition is in the next code cell. It uses `segmentation_models_pytorch.Unet` with a ResNet-34 encoder, ImageNet encoder weights, 3-channel RGB input, and a single binary logit output channel (`sigmoid > 0.5` at inference time).


In [8]:
# Model configuration and factory
MODEL_CONFIG = {
    'encoder_name': 'resnet34',
    'encoder_weights': 'imagenet',
    'in_channels': 3,
    'classes': 1,  # Single binary channel (sigmoid)
}

def create_model(
    encoder_name: str = MODEL_CONFIG['encoder_name'],
    encoder_weights: str = MODEL_CONFIG['encoder_weights'],
    in_channels: int = MODEL_CONFIG['in_channels'],
    classes: int = MODEL_CONFIG['classes'],
) -> smp.Unet:
    """Create smp.Unet with binary output (1 channel logits, no activation)."""
    model = smp.Unet(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=in_channels,
        classes=classes,
        activation=None,  # Raw logits for BCE-with-logits loss
    )
    return model


In [9]:
# Smoke test: smp U-Net forward pass
model_test = create_model().to(device)
total_params = sum(p.numel() for p in model_test.parameters())
encoder_params = sum(p.numel() for p in model_test.encoder.parameters())
decoder_params = total_params - encoder_params

print(f"smp U-Net total parameters:   {total_params:,}")
print(f"  Encoder (ResNet-34):         {encoder_params:,}")
print(f"  Decoder + head:              {decoder_params:,}")

x_test = torch.randn(1, 3, 1024, 1024, device=device)
with torch.no_grad():
    out_test = model_test(x_test)
print(f"Input shape:  {x_test.shape}")
print(f"Output shape: {out_test.shape}")
assert out_test.shape == (1, 1, 1024, 1024), f"Expected (1, 1, 1024, 1024), got {out_test.shape}"
print("✓ smp U-Net forward pass OK")

del model_test, x_test, out_test
torch.cuda.empty_cache()

smp U-Net total parameters:   24,436,369
  Encoder (ResNet-34):         21,284,672
  Decoder + head:              3,151,697
Input shape:  torch.Size([1, 3, 1024, 1024])
Output shape: torch.Size([1, 1, 1024, 1024])
✓ smp U-Net forward pass OK


## Part 4: Training Pipeline

Complete training system with:
- Dynamic DataLoader worker calculation (based on available RAM)
- Binary Segmentation Metrics (Accuracy, Precision, Recall, F1, ROC-AUC, Dice)
- Augmentation pipelines (train-specific)
- Optimizer: AdamW with weight decay
- LR Scheduler: Linear warmup (5 epochs) -> Cosine annealing
- Checkpointing: saves best model (by Val F1) + last model
- TensorBoard logging

### Training Workflow

1. Load data (train/val/test splits grouped by biopsia)
2. Create model, optimizer, scheduler
3. Train for N epochs:
   - Forward pass on batch
   - Compute loss (BCEWithLogits + Dice for binary segmentation)
   - Backward pass + gradient clipping
   - Optimizer step
   - Validate after each epoch (compute binary metrics on val set)
   - Save checkpoint if best model found
4. Final evaluation on test set
5. Save metrics report

In [10]:
class MeanIoUMetric:
    """IoU for binary segmentation with 1-channel sigmoid output."""

    def __init__(self, threshold: float = 0.5):
        self.threshold = threshold
        self.reset()

    def reset(self):
        self.intersection = 0
        self.union = 0

    def update(self, logits: torch.Tensor, target: torch.Tensor):
        """
        Args:
            logits:  [B, 1, H, W] raw logits
            target:  [B, H, W] int64 binary labels
        """
        probs = torch.sigmoid(logits).squeeze(1)           # [B, H, W]
        pred_labels = (probs > self.threshold).long()

        pred_np = pred_labels.cpu().numpy()
        tgt_np  = target.cpu().numpy()

        self.intersection += np.logical_and(pred_np == 1, tgt_np == 1).sum()
        self.union        += np.logical_or(pred_np == 1, tgt_np == 1).sum()

    def compute(self) -> float:
        if self.union == 0:
            return 0.0
        return float(self.intersection / self.union)

In [11]:
class BinarySegmentationMetrics:
    """Accuracy, Precision, Recall, F1, Dice, and ROC-AUC for binary segmentation with sigmoid output."""

    def __init__(self, max_auc_pixels: int = 100000, threshold: float = 0.5):
        """Store subsampled pixels to prevent memory explosion with large images."""
        self.max_auc_pixels = max_auc_pixels
        self.threshold = threshold
        self.reset()

    def reset(self):
        self.tp = 0
        self.tn = 0
        self.fp = 0
        self.fn = 0
        
        self.all_probs = []
        self.all_targets = []
        self.total_auc_pixels = 0

    def update(self, pred: torch.Tensor, target: torch.Tensor):
        """Accumulate confusion matrix and subsampled pixel predictions.
        
        Args:
            pred: [B, 1, H, W] raw logits from smp model
            target: [B, H, W] int64 binary labels {0, 1}
        """
        prob_pos = torch.sigmoid(pred).squeeze(1)          # [B, H, W]
        pred_labels = (prob_pos > self.threshold).long()

        pred_labels = pred_labels.cpu().numpy()
        target = target.cpu().numpy()
        prob_pos = prob_pos.cpu().numpy()

        pred_flat = pred_labels.flatten()
        target_flat = target.flatten()
        prob_flat = prob_pos.flatten()

        self.tp += np.logical_and(target_flat == 1, pred_flat == 1).sum()
        self.tn += np.logical_and(target_flat == 0, pred_flat == 0).sum()
        self.fp += np.logical_and(target_flat == 0, pred_flat == 1).sum()
        self.fn += np.logical_and(target_flat == 1, pred_flat == 0).sum()

        # Subsample fixed amount per batch to ensure uniform coverage across all batches
        k_samples = min(1000, len(prob_flat))
        idx = np.random.choice(len(prob_flat), size=k_samples, replace=False)
        self.all_probs.extend(prob_flat[idx])
        self.all_targets.extend(target_flat[idx])
        
        # Reservoir capping to max_auc_pixels
        if len(self.all_probs) > self.max_auc_pixels:
            keep_idx = np.random.choice(len(self.all_probs), size=self.max_auc_pixels, replace=False)
            self.all_probs = [self.all_probs[i] for i in keep_idx]
            self.all_targets = [self.all_targets[i] for i in keep_idx]
            
        self.total_auc_pixels += len(prob_flat)

    def compute(self) -> dict:
        """Return accuracy, precision, recall, f1, dice, roc_auc."""
        metrics = {}

        total = self.tp + self.tn + self.fp + self.fn
        if total > 0:
            metrics['accuracy'] = float((self.tp + self.tn) / total)
        else:
            metrics['accuracy'] = 0.0

        if (self.tp + self.fp) > 0:
            metrics['precision'] = float(self.tp / (self.tp + self.fp))
        else:
            metrics['precision'] = 0.0

        if (self.tp + self.fn) > 0:
            metrics['recall'] = float(self.tp / (self.tp + self.fn))
        else:
            metrics['recall'] = 0.0

        if (metrics['precision'] + metrics['recall']) > 0:
            metrics['f1'] = float(
                2 * (metrics['precision'] * metrics['recall']) / 
                (metrics['precision'] + metrics['recall'])
            )
        else:
            metrics['f1'] = 0.0

        if (2 * self.tp + self.fp + self.fn) > 0:
            metrics['dice_metric'] = float(
                (2 * self.tp) / (2 * self.tp + self.fp + self.fn + 1e-8)
            )
        else:
            metrics['dice_metric'] = 0.0

        if len(self.all_targets) > 0 and len(set(self.all_targets)) > 1:
            try:
                from sklearn.metrics import roc_auc_score
                auc = roc_auc_score(self.all_targets, self.all_probs)
                metrics['roc_auc'] = float(auc)
            except Exception as e:
                print(f"Warning: Could not compute ROC-AUC: {e}")
                metrics['roc_auc'] = 0.0
        else:
            metrics['roc_auc'] = 0.0

        return metrics

In [12]:
def get_transforms(size: int = 1024):
    """Return augmentation pipelines for train/val.
    
    Note: Reinhard and Z-score normalization happen in GlomeruliDataset.__getitem__,
    not here. These transforms run on normalized data.
    """
    train_augment = A.Compose([
        # Geometric augmentations — applied to both image AND mask
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.75),
        A.Transpose(p=0.5),
        A.Rotate(limit=15, p=0.3),
        A.ShiftScaleRotate(scale_limit=0.15, rotate_limit=15, shift_limit=0.1, p=0.5),
        
        # Elastic deformations — simulate tissue preparation artifacts
        A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
        
        # Color augmentations — simulate stain variation between slides
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.3),
        
        # Noise — simulate scanner artifacts
        A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
        
        # Regularization via occlusion (use fill=128 to avoid NaN in BatchNorm)
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(32, 64),
            hole_width_range=(32, 64),
            fill=128,
            p=0.2,
        ),
    ], additional_targets={'mask': 'mask'})
    
    val_transform = A.Compose([])
    return train_augment, val_transform


# Visual Data Pipeline — Preprocessing Stages

This section shows how the data transforms through each preprocessing step:
1. **Original** - Raw tile from histological image
2. **Reinhard Normalization** - Color normalization to match reference stain
3. **Z-score Normalization** - Per-channel standardization (visualized by denormalizing for display)
4. **Binary Mask** - Ground truth segmentation (white=glomerulus, black=background)

In [ ]:
def visualize_preprocessing_pipeline(tile_path, mask_path, reinhard_norm=None, channel_means=None, channel_stds=None):
    """
    Show one tile progressing through the preprocessing pipeline.
    
    Args:
        tile_path: Path to input tile image
        mask_path: Path to mask image
        reinhard_norm: ReinhardNormalize instance
        channel_means: List of channel means for Z-score
        channel_stds: List of channel stds for Z-score
    """
    import matplotlib.pyplot as plt
    import cv2
    from PIL import Image
    
    # Load original
    img_rgb = cv2.cvtColor(cv2.imread(tile_path), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    # Step 1: Original
    img_original = img_rgb.copy()
    
    # Step 2: Reinhard normalization
    img_reinhard = img_rgb.copy()
    if reinhard_norm:
        img_reinhard = reinhard_norm(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
        img_reinhard = cv2.cvtColor(img_reinhard, cv2.COLOR_BGR2RGB)
    
    # Step 3: Z-score (simulate the tensor preprocessing)
    img_zscore = img_reinhard.astype(np.float32) / 255.0
    if channel_means and channel_stds:
        img_zscore_norm = img_zscore.copy()
        for c in range(3):
            img_zscore_norm[..., c] = (img_zscore_norm[..., c] - channel_means[c]) / (channel_stds[c] + 1e-8)
        # Denormalize for visualization
        img_zscore_vis = img_zscore_norm.copy()
        for c in range(3):
            # Visualize Z-scores by scaling to 0-1 range (NOT denormalizing)
            # Values around -3 to +3 are typical; shift and scale for visualization
            img_zscore_vis = np.clip((img_zscore_norm + 3) / 6, 0, 1)
    
    # Step 4: Mask
    mask_binary = (mask > 0).astype(np.uint8)
    
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    
    # Original
    axes[0].imshow(img_original)
    axes[0].set_title("1. Original", fontweight='bold')
    axes[0].text(0.5, -0.15, f"RGB uint8, μ={img_original.mean():.1f}", 
                ha='center', transform=axes[0].transAxes, fontsize=9)
    axes[0].axis('off')
    
    # Reinhard
    axes[1].imshow(img_reinhard)
    axes[1].set_title("2. Reinhard Normalized", fontweight='bold')
    axes[1].text(0.5, -0.15, "Color normalization (LAB space)", 
                ha='center', transform=axes[1].transAxes, fontsize=9)
    axes[1].axis('off')
    
    # Z-score
    axes[2].imshow(img_zscore_vis)
    axes[2].set_title("3. Z-score Normalized", fontweight='bold')
    axes[2].text(0.5, -0.15, f"Per-channel standardization", 
                ha='center', transform=axes[2].transAxes, fontsize=9)
    axes[2].axis('off')
    
    # Mask
    mask_display = np.stack([mask_binary, mask_binary, mask_binary], axis=-1) * 255
    axes[3].imshow(mask_display)
    axes[3].set_title("4. Binary Mask", fontweight='bold')
    axes[3].text(0.5, -0.15, f"Ground truth (1={mask_binary.sum()} pixels)", 
                ha='center', transform=axes[3].transAxes, fontsize=9)
    axes[3].axis('off')
    
    plt.suptitle(f"Data Preprocessing Pipeline: {Path(tile_path).name}", 
                fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    plt.close()

# Data Augmentation Effects

Each augmentation is applied individually to show its effect on the tile and mask separately. 
This helps understand how each transformation preserves the glomerulus regions while adding robustness to the model.

In [ ]:
def visualize_augmentations(tile_path, mask_path, n_rows=3, n_per_row=4):
    """
    Apply individual augmentations to a tile and visualize each one.
    
    Args:
        tile_path: Path to tile image
        mask_path: Path to mask image
        n_rows: Number of rows in grid
        n_per_row: Number of augmentations per row
    """
    import matplotlib.pyplot as plt
    import cv2
    import albumentations as A
    
    # Load data
    img_rgb = cv2.cvtColor(cv2.imread(tile_path), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask_binary = (mask > 0).astype(np.uint8)
    
    # Define individual augmentations
    augmentations = [
        ("Original", A.NoOp()),
        ("HorizontalFlip", A.HorizontalFlip(p=1.0)),
        ("VerticalFlip", A.VerticalFlip(p=1.0)),
        ("RandomRotate90", A.RandomRotate90(p=1.0)),
        ("Transpose", A.Transpose(p=1.0)),
        ("Rotate(±15°)", A.Rotate(limit=15, p=1.0)),
        ("ShiftScaleRotate", A.ShiftScaleRotate(scale_limit=0.15, rotate_limit=15, shift_limit=0.1, p=1.0)),
        ("ElasticTransform", A.ElasticTransform(alpha=120, sigma=6, p=1.0)),
        ("GridDistortion", A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0)),
        ("ColorJitter", A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=1.0)),
        ("GaussNoise", A.GaussNoise(std_range=(0.01, 0.05), p=1.0)),
        ("CoarseDropout", A.CoarseDropout(num_holes_range=(1,8), hole_height_range=(32,64), hole_width_range=(32,64), fill=128, p=1.0)),
    ]
    
    n_total = min(len(augmentations), n_rows * n_per_row)
    
    fig, axes = plt.subplots(n_rows, n_per_row, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, (aug_name, augmentor) in enumerate(augmentations[:n_total]):
        # Apply augmentation
        augmented = augmentor(image=img_rgb, mask=mask_binary)
        img_aug = augmented['image']
        mask_aug = augmented['mask']
        
        # Create overlay
        overlay = img_aug.copy().astype(float)
        overlay[mask_aug == 1] = [255, 0, 0]
        overlay = overlay.astype(np.uint8)
        
        alpha = 0.5
        blended = cv2.addWeighted(img_aug, 1-alpha, overlay, alpha, 0)
        
        axes[idx].imshow(blended)
        axes[idx].set_title(aug_name, fontsize=10, fontweight='bold')
        axes[idx].axis('off')
    
    # Hide unused subplots
    for idx in range(n_total, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle("Data Augmentation Effects (mask overlay in red)", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
if Path('Salidas/Tiles_UNet').exists():
    # Get a sample positive tile for pipeline visualization
    ds_viz = GlomeruliDataset('Salidas/Tiles_UNet', split='train', 
                             reinhard_norm=None, channel_means=None, channel_stds=None)
    positive_flags = ds_viz.get_positive_flags()
    
    # Compute normalization parameters from train data
    print("Computing Reinhard normalization template...")
    train_image_paths = ds_viz.image_paths  # Get all image paths
    
    try:
        reinhard_stats = ReinhardNormalize.compute_template_stats(train_image_paths, n_samples=min(200, len(train_image_paths)))
        reinhard_norm = ReinhardNormalize(reinhard_stats)
    except Exception as e:
        print(f"Warning: Could not compute Reinhard stats: {e}")
        reinhard_norm = None
        reinhard_stats = None
    
    try:
        channel_means, channel_stds = compute_channel_stats(train_image_paths, n_samples=min(200, len(train_image_paths)))
    except Exception as e:
        print(f"Warning: Could not compute channel stats: {e}")
        channel_means = None
        channel_stds = None
    
    # Find first positive tile from a biopsia starting with "BR-"
    sample_idx = None
    for i, flag in enumerate(positive_flags):
        if flag:
            img_path_str = str(ds_viz.image_paths[i])
            if 'BR-' in img_path_str:  # Look for biopsias starting with BR-
                sample_idx = i
                break
    if sample_idx is None:
        # Fallback: use any positive tile if no BR- found
        sample_idx = [i for i, flag in enumerate(positive_flags) if flag][0]
    
    sample_tile = ds_viz.image_paths[sample_idx]
    sample_mask = ds_viz._get_mask_path(sample_tile)
    
    visualize_preprocessing_pipeline(sample_tile, sample_mask, reinhard_norm=reinhard_norm, channel_means=channel_means, channel_stds=channel_stds)
    visualize_augmentations(sample_tile, sample_mask)


# Glomeruli on Tile Borders — Overlapping from Tiling

When tiles are extracted from the large WSI (Whole Slide Image), some glomeruli may be cut off at tile boundaries.
This visualization identifies tiles where glomeruli touch the border, which indicates partial coverage that continues in adjacent tiles.

In [ ]:
def visualize_border_glomeruli(images_dir, n_tiles=4, border_margin=30):
    """
    Visualize tiles where glomeruli touch the border (partial glomeruli).
    
    Args:
        images_dir: Root directory with tile data
        n_tiles: Number of border tiles to show
        border_margin: Pixel distance from edge to consider as "border"
    """
    import matplotlib.pyplot as plt
    import cv2
    import matplotlib.patches as patches
    from pathlib import Path
    
    ds = GlomeruliDataset(images_dir, split='train', reinhard_norm=None, 
                          channel_means=None, channel_stds=None)
    positive_flags = ds.get_positive_flags()
    
    # Find tiles with border glomeruli
    border_tiles = []
    for idx, has_positive in enumerate(positive_flags):
        if not has_positive:
            continue
        
        mask_path = ds._get_mask_path(ds.image_paths[idx])
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask_binary = (mask > 0)
        
        # Check if mask touches border
        border_pixels = (mask_binary[:border_margin, :].any() or 
                        mask_binary[-border_margin:, :].any() or 
                        mask_binary[:, :border_margin].any() or 
                        mask_binary[:, -border_margin:].any())
        
        if border_pixels:
            border_tiles.append(idx)
    
    if len(border_tiles) == 0:
        print("No tiles with border glomeruli found")
        return
    
    # Show first n_tiles border tiles
    import random
    random.seed(42)
    selected = random.sample(border_tiles, min(n_tiles, len(border_tiles)))
    
    fig, axes = plt.subplots(1, len(selected), figsize=(5*len(selected), 5))
    if len(selected) == 1:
        axes = np.array([axes])
    
    for col, idx in enumerate(selected):
        image_path = ds.image_paths[idx]
        mask_path = ds._get_mask_path(image_path)
        
        img_rgb = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask_binary = (mask > 0).astype(np.uint8)
        
        # Overlay
        overlay = img_rgb.copy().astype(float)
        overlay[mask_binary == 1] = [255, 0, 0]
        overlay = overlay.astype(np.uint8)
        alpha = 0.4
        blended = cv2.addWeighted(img_rgb, 1-alpha, overlay, alpha, 0)
        
        axes[col].imshow(blended)
        
        # Draw red rectangles at borders
        h, w = img_rgb.shape[:2]
        rect_top = patches.Rectangle((0, 0), w, border_margin, linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
        rect_bottom = patches.Rectangle((0, h-border_margin), w, border_margin, linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
        rect_left = patches.Rectangle((0, 0), border_margin, h, linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
        rect_right = patches.Rectangle((w-border_margin, 0), border_margin, h, linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
        
        axes[col].add_patch(rect_top)
        axes[col].add_patch(rect_bottom)
        axes[col].add_patch(rect_left)
        axes[col].add_patch(rect_right)
        
        axes[col].set_title(f"Tile {idx}\n(border marked in red)", fontsize=10, fontweight='bold')
        axes[col].text(0.5, -0.1, "⚠ Glomerulus continues in adjacent tile", 
                      ha='center', transform=axes[col].transAxes, fontsize=9, color='red')
        axes[col].axis('off')
    
    plt.suptitle(f"Glomeruli on Tile Borders (margin={border_margin}px)", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
if Path('Salidas/Tiles_UNet').exists():
    visualize_border_glomeruli('Salidas/Tiles_UNet', n_tiles=4, border_margin=30)

In [13]:
def train_epoch(
    model: nn.Module,
    dataloader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    scaler: GradScaler = None,
    epoch: int = 0,
    total_epochs: int = 1,
    accum_steps: int = 1,
) -> float:
    """Train one epoch with tqdm progress, per-batch timing, AMP (FP16), and gradient accumulation."""
    model.train()
    total_loss = 0.0
    num_batches = 0
    running_dice = 0.0
    
    epoch_start = time.time()
    
    pbar = tqdm(
        dataloader,
        desc=f"Epoch {epoch+1}/{total_epochs} [train]",
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )
    
    for batch_idx, (images, masks) in enumerate(pbar):
        t_data_end = time.time()

        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)

        t_transfer_end = time.time()

        # Zero gradients only at the start of accumulation cycle, not every batch
        if batch_idx % accum_steps == 0:
            optimizer.zero_grad(set_to_none=True)

        t_forward_start = time.time()
        if scaler is not None:
            with autocast():
                logits = model(images)
                loss = criterion(logits, masks)
                # Scale loss by accumulation steps
                loss = loss / accum_steps
            scaler.scale(loss).backward()
            
            # Only unscale, clip, and step every accum_steps batches or at end of epoch
            if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(dataloader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, masks)
            # Scale loss by accumulation steps
            loss = loss / accum_steps
            loss.backward()
            
            # Only clip and step every accum_steps batches or at end of epoch
            if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(dataloader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        t_batch_end = time.time()

        # Store unscaled loss for logging (accumulation scaling was only for gradient)
        batch_loss = loss.item() * accum_steps
        total_loss += batch_loss
        num_batches += 1

        # Quick inline Dice for the progress bar (no grad, cheap)
        with torch.no_grad():
            prob = torch.sigmoid(logits).squeeze(1)
            pred = (prob > 0.5).float()
            tgt  = masks.float()
            inter = (pred * tgt).sum()
            denom = pred.sum() + tgt.sum()
            batch_dice = float((2 * inter + 1e-6) / (denom + 1e-6))
        running_dice = (running_dice * (num_batches - 1) + batch_dice) / num_batches

        pbar.set_postfix({
            'loss': f'{batch_loss:.4f}',
            'avg_loss': f'{total_loss/num_batches:.4f}',
            'dice': f'{batch_dice:.3f}',
            's/batch': f'{t_batch_end - t_forward_start:.2f}',
        })

        # Print first batch immediately so we know training started
        if batch_idx == 0:
            print(f"\n  [Epoch {epoch+1}] First batch: loss={batch_loss:.4f}, dice={batch_dice:.3f}")

    epoch_elapsed = time.time() - epoch_start
    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    
    return avg_loss


In [14]:
def eval_epoch(
    model: nn.Module,
    dataloader,
    criterion: nn.Module,
    device: torch.device,
    split_name: str = 'val',
    threshold: float = 0.5,
) -> Tuple[float, dict]:
    """Evaluate one epoch with tqdm progress. Returns loss and binary segmentation metrics."""
    model.eval()
    total_loss = 0.0
    num_batches = 0
    seg_metrics = BinarySegmentationMetrics(threshold=threshold)
    iou_metric  = MeanIoUMetric(threshold=threshold)

    pbar = tqdm(dataloader, desc=f"  [{split_name}]", unit="batch",
                dynamic_ncols=True, leave=False)

    with torch.no_grad():
        for images, masks in pbar:
            images = images.to(device, non_blocking=True)
            masks  = masks.to(device, non_blocking=True)

            with autocast():
                logits = model(images)
                loss   = criterion(logits, masks)

            total_loss += loss.item()
            num_batches += 1

            seg_metrics.update(logits, masks)
            iou_metric.update(logits, masks)

            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    metrics  = seg_metrics.compute()
    metrics['mean_iou'] = iou_metric.compute()

    return avg_loss, metrics


In [15]:
def compute_dataloader_workers(batch_size: int = 8, max_workers: int = 8) -> int:
    """Heuristic: calculate optimal DataLoader workers based on available CPU cores."""
    import os

    n_cpus = os.cpu_count() or 4
    n_workers = min(max(batch_size // 2, 2), n_cpus, max_workers)
    return n_workers


In [16]:
def find_best_threshold(
    all_probs: torch.Tensor,
    all_targets: torch.Tensor,
    thresholds: list = None,
) -> Tuple[float, dict]:
    """
    Find the best threshold for binary segmentation based on F1 score.
    
    Args:
        all_probs:   [N_samples] predicted probabilities (0-1)
        all_targets: [N_samples] ground truth binary labels (0-1)
        thresholds:  list of thresholds to search. Default: [0.10, 0.15, 0.20, ..., 0.60]
    
    Returns:
        (best_threshold, metrics_dict) where metrics_dict contains F1, precision, recall for best threshold
    """
    if thresholds is None:
        thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60]
    
    all_probs = all_probs.cpu().float()
    all_targets = all_targets.cpu().float()
    
    best_f1 = 0.0
    best_threshold = 0.5
    best_metrics = {}
    
    for thresh in thresholds:
        pred = (all_probs > thresh).float()
        
        tp = (pred * all_targets).sum().item()
        fp = (pred * (1 - all_targets)).sum().item()
        fn = ((1 - pred) * all_targets).sum().item()
        
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * tp / (2 * tp + fp + fn + 1e-8)
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh
            best_metrics = {
                'f1': f1,
                'precision': precision,
                'recall': recall,
                'tp': int(tp),
                'fp': int(fp),
                'fn': int(fn),
            }
    
    return best_threshold, best_metrics


def audit_mask_distribution(
    datasets: dict,
    expected_values=None,
    almost_full_threshold: float = 0.80,
):
    """Print a quantitative mask audit before training."""
    print("\n--- Mask Audit Report (pre-training) ---")
    if expected_values is None:
        print("Expected raw mask values: not enforced")
    else:
        expected_values = frozenset(expected_values)
        print(f"Expected raw mask values: {sorted(expected_values)}")
    print(f"Almost-full tile threshold: positive_area_ratio >= {almost_full_threshold:.2f}")

    all_unexpected_values = set()
    audit_summary = {}
    percentiles = [0, 1, 5, 25, 50, 75, 95, 99, 100]

    for split_name, dataset in datasets.items():
        tile_records = []
        value_pixel_counts = {}
        value_tile_counts = {}
        slide_stats = {}

        for idx, mask_path in enumerate(dataset.mask_paths):
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                warnings.warn(f"Could not read mask during audit: {mask_path}")
                continue

            image_path = dataset.image_paths[idx]
            slide = _slide_name_from_image_path(image_path, dataset.images_dir)
            get_tile_metadata = getattr(dataset, 'get_tile_metadata', None)
            if callable(get_tile_metadata):
                try:
                    meta = get_tile_metadata(idx)
                    slide = meta.get('slide') or slide
                except Exception:
                    pass

            unique_vals, counts = np.unique(mask, return_counts=True)
            unique_vals = [int(v) for v in unique_vals]
            counts = [int(c) for c in counts]
            for val, count in zip(unique_vals, counts):
                value_pixel_counts[val] = value_pixel_counts.get(val, 0) + count
                value_tile_counts[val] = value_tile_counts.get(val, 0) + 1

            if expected_values is not None:
                unexpected_vals = set(unique_vals) - set(expected_values)
                all_unexpected_values.update(unexpected_vals)
            else:
                unexpected_vals = set()

            pos_mask = mask > 0
            pos_pixels = int(pos_mask.sum())
            total_pixels = int(mask.size)
            neg_pixels = total_pixels - pos_pixels
            pos_ratio = pos_pixels / total_pixels if total_pixels else 0.0
            is_positive = pos_pixels > 0
            is_almost_full = pos_ratio >= almost_full_threshold

            tile_records.append({
                'slide': slide,
                'pos_pixels': pos_pixels,
                'neg_pixels': neg_pixels,
                'pos_ratio': pos_ratio,
                'is_positive': is_positive,
                'is_empty': not is_positive,
                'is_almost_full': is_almost_full,
                'unique_values': unique_vals,
                'unexpected_values': sorted(unexpected_vals),
                'mask_path': str(mask_path),
            })

            st = slide_stats.setdefault(slide, {
                'tiles': 0,
                'positive_tiles': 0,
                'empty_tiles': 0,
                'almost_full_tiles': 0,
                'pos_pixels': 0,
                'neg_pixels': 0,
                'values': {},
            })
            st['tiles'] += 1
            st['positive_tiles'] += int(is_positive)
            st['empty_tiles'] += int(not is_positive)
            st['almost_full_tiles'] += int(is_almost_full)
            st['pos_pixels'] += pos_pixels
            st['neg_pixels'] += neg_pixels
            for val, count in zip(unique_vals, counts):
                st['values'][val] = st['values'].get(val, 0) + count

        if not tile_records:
            warnings.warn(f"No readable masks found for split '{split_name}' during audit.")
            audit_summary[split_name] = {'tile_records': [], 'slide_stats': {}, 'value_pixel_counts': {}}
            continue

        ratios = np.array([r['pos_ratio'] for r in tile_records], dtype=np.float64)
        positive_ratios = np.array([r['pos_ratio'] for r in tile_records if r['is_positive']], dtype=np.float64)
        all_pct = {p: float(np.percentile(ratios, p)) for p in percentiles}
        pos_pct = {p: float(np.percentile(positive_ratios, p)) for p in percentiles} if positive_ratios.size else {}

        n_tiles = len(tile_records)
        n_positive = sum(r['is_positive'] for r in tile_records)
        n_empty = sum(r['is_empty'] for r in tile_records)
        n_almost_full = sum(r['is_almost_full'] for r in tile_records)
        pos_pixels_total = sum(r['pos_pixels'] for r in tile_records)
        neg_pixels_total = sum(r['neg_pixels'] for r in tile_records)
        total_pixels = pos_pixels_total + neg_pixels_total
        raw_values_seen = sorted(value_pixel_counts)
        unexpected_seen = sorted(set(raw_values_seen) - set(expected_values)) if expected_values is not None else []
        rare_values = {
            val: {'tiles': value_tile_counts[val], 'pixels': value_pixel_counts[val]}
            for val in raw_values_seen
            if val != 0 and value_tile_counts[val] <= max(1, int(0.01 * n_tiles))
        }

        print(f"\n[{split_name.upper()}]")
        print(f"  Tiles: {n_tiles:,} | positive: {n_positive:,} | empty: {n_empty:,} | almost full: {n_almost_full:,}")
        print(f"  Pixels: positive={pos_pixels_total:,} negative={neg_pixels_total:,} "
              f"positive_ratio={(pos_pixels_total / total_pixels if total_pixels else 0):.6f}")
        print("  Positive area ratio percentiles (all tiles): " +
              ", ".join(f"p{p}={all_pct[p]:.6f}" for p in percentiles))
        if pos_pct:
            print("  Positive area ratio percentiles (positive tiles only): " +
                  ", ".join(f"p{p}={pos_pct[p]:.6f}" for p in percentiles))
        else:
            print("  Positive area ratio percentiles (positive tiles only): no positive tiles")
        print(f"  Raw mask values seen: {raw_values_seen}")
        print(f"  Raw mask pixel counts: {dict(sorted(value_pixel_counts.items()))}")
        print(f"  Raw mask tile counts: {dict(sorted(value_tile_counts.items()))}")
        if unexpected_seen:
            warnings.warn(f"Unexpected raw mask values in {split_name}: {unexpected_seen}")
        if rare_values:
            print(f"  Rare non-zero values (<=1% of tiles): {rare_values}")

        print("  Per-slide summary:")
        for slide, st in sorted(slide_stats.items()):
            slide_total = st['pos_pixels'] + st['neg_pixels']
            slide_pos_ratio = st['pos_pixels'] / slide_total if slide_total else 0.0
            flags = []
            if st['positive_tiles'] == 0:
                flags.append('NO_POSITIVES')
            if st['almost_full_tiles'] > 0:
                flags.append('ALMOST_FULL_TILES')
            flag_txt = f" [{' | '.join(flags)}]" if flags else ""
            print(
                f"    {slide}: tiles={st['tiles']:,}, positive_tiles={st['positive_tiles']:,}, "
                f"empty_tiles={st['empty_tiles']:,}, almost_full={st['almost_full_tiles']:,}, "
                f"pos_pixels={st['pos_pixels']:,}, neg_pixels={st['neg_pixels']:,}, "
                f"pos_ratio={slide_pos_ratio:.6f}, values={sorted(st['values'])}{flag_txt}"
            )

        audit_summary[split_name] = {
            'tile_records': tile_records,
            'slide_stats': slide_stats,
            'value_pixel_counts': value_pixel_counts,
            'value_tile_counts': value_tile_counts,
            'unexpected_values': unexpected_seen,
            'rare_values': rare_values,
            'percentiles_all_tiles': all_pct,
            'percentiles_positive_tiles': pos_pct,
        }

    if all_unexpected_values:
        warnings.warn(f"Unexpected raw mask values found across splits: {sorted(all_unexpected_values)}")
    print("\n--- End Mask Audit Report ---")
    return audit_summary


def train_unet(
    epochs: int = 50,
    batch_size: int = 8,
    lr: float = 1e-3,
    images_dir: str = 'Salidas/Tiles_UNet',
    output_dir: str = 'checkpoints',
    num_workers: int = None,
    seed: int = 42,
    warmup_epochs: int = 5,
    use_amp: bool = True,
    accum_steps: int = 1,
    use_gradient_checkpointing: bool = False,
):
    """Complete training pipeline with smp.Unet, WeightedRandomSampler, and tqdm logging."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    run_name = f"unet_binary_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    log_dir = output_dir / run_name
    writer = SummaryWriter(str(log_dir))

    print(f"Run: {run_name}")
    print(f"Device: {device}")
    print(f"AMP (FP16): {use_amp and device.type == 'cuda'}")
    print(f"Output dir: {output_dir}")

    print("\n--- Data Loading Pipeline ---")
    print("Step 1: Computing grouped split by biopsia...")
    train_biopsias, val_biopsias, test_biopsias, biopsias_dict = split_biopsias(
        images_dir=images_dir,
        train_size=0.70,
        val_size=0.15,
        seed=seed,
    )
    print(f"  Train biopsies: {len(train_biopsias)}")
    print(f"  Val biopsies: {len(val_biopsias)}")
    print(f"  Test biopsies: {len(test_biopsias)}")

    print("\nStep 2: Collecting train image paths...")
    train_image_paths = []
    for biopsia in train_biopsias:
        train_image_paths.extend(biopsias_dict[biopsia])
    print(f"  Train tiles: {len(train_image_paths)}")

    print("\nStep 3: Computing Reinhard stain normalization template...")
    try:
        reinhard_stats = ReinhardNormalize.compute_template_stats(train_image_paths, n_samples=200)
        reinhard_norm = ReinhardNormalize(reinhard_stats)
        print(f"  Template stats computed from ~{min(200, len(train_image_paths))} samples")
        print(f"    L: mean={reinhard_stats['mean_L']:.1f}, std={reinhard_stats['std_L']:.1f}")
        print(f"    a: mean={reinhard_stats['mean_a']:.1f}, std={reinhard_stats['std_a']:.1f}")
        print(f"    b: mean={reinhard_stats['mean_b']:.1f}, std={reinhard_stats['std_b']:.1f}")
    except Exception as e:
        print(f"  Warning: Could not compute Reinhard stats: {e}")
        reinhard_norm = None
        reinhard_stats = None

    print("\nStep 4: Computing per-channel Z-score normalization stats...")
    try:
        channel_means, channel_stds = compute_channel_stats(train_image_paths, n_samples=200)
        print(f"  Channel means (RGB): {[f'{m:.4f}' for m in channel_means]}")
        print(f"  Channel stds (RGB):  {[f'{s:.4f}' for s in channel_stds]}")
    except Exception as e:
        print(f"  Warning: Could not compute channel stats: {e}")
        channel_means = None
        channel_stds = None

    print("\nStep 5: Creating augmentation pipelines...")
    train_transforms, val_transforms = get_transforms()

    if num_workers is None:
        num_workers = compute_dataloader_workers(batch_size=batch_size)
        print(f"  Auto-calculated DataLoader workers: {num_workers}")
    else:
        print(f"  Using specified DataLoader workers: {num_workers}")

    print("\nStep 5b: Creating dataloaders with pre-computed splits and WeightedRandomSampler...")
    train_loader, val_loader, test_loader = create_dataloaders(
        images_dir=images_dir,
        batch_size=batch_size,
        num_workers=num_workers,
        seed=seed,
        train_transforms=train_transforms,
        val_transforms=val_transforms,
        reinhard_norm=reinhard_norm,
        channel_means=channel_means,
        channel_stds=channel_stds,
    )

    print(f"  Train: {len(train_loader.dataset)} tiles | "
          f"Val: {len(val_loader.dataset)} tiles | "
          f"Test: {len(test_loader.dataset)} tiles")

    audit_summary = audit_mask_distribution({
        'train': train_loader.dataset,
        'val': val_loader.dataset,
        'test': test_loader.dataset,
    })

    print("\nCreating smp.Unet model with ResNet-34 encoder...")
    model = create_model().to(device)
    total_params = sum(p.numel() for p in model.parameters())
    
    # Optional: Enable gradient checkpointing to save memory
    if use_gradient_checkpointing:
        if hasattr(model, 'encoder') and hasattr(model.encoder, 'set_grad_checkpointing'):
            model.encoder.set_grad_checkpointing(True)
            print(f"  Gradient checkpointing enabled for encoder")
        else:
            print(f"  Warning: Model does not support gradient checkpointing")
    print(f"Model parameters: {total_params:,}")

    criterion = BinaryCombinedLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    scaler = GradScaler() if (use_amp and device.type == 'cuda') else None
    if scaler:
        print("AMP (FP16) enabled — GradScaler initialized")

    warmup_epochs = min(warmup_epochs, max(0, epochs - 1))
    cosine_epochs = max(1, epochs - warmup_epochs)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[
            LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=max(1, warmup_epochs)),
            CosineAnnealingLR(optimizer, T_max=cosine_epochs, eta_min=1e-6),
        ],
        milestones=[warmup_epochs],
    )

    threshold = 0.5
    best_val_f1 = -1.0
    train_history = []

    print("\nStarting training...")
    for epoch in range(epochs):
        print(f"\n{'='*60}")
        print(f"Epoch [{epoch+1}/{epochs}]")
        print(f"{'='*60}")

        train_loss = train_epoch(
            model, train_loader, criterion, optimizer, device,
            scaler=scaler, epoch=epoch, total_epochs=epochs,
            accum_steps=accum_steps,
        )

        val_loss, val_metrics = eval_epoch(model, val_loader, criterion, device, split_name='val')
        print(f"Val Loss: {val_loss:.4f}")
        print(f"Val Metrics:")
        print(f"  Accuracy:  {val_metrics['accuracy']:.4f}")
        print(f"  Precision: {val_metrics['precision']:.4f}")
        print(f"  Recall:    {val_metrics['recall']:.4f}")
        print(f"  F1:        {val_metrics['f1']:.4f}")
        print(f"  Dice:      {val_metrics.get('dice_metric', 0.0):.4f}")
        print(f"  mIoU:      {val_metrics.get('mean_iou', 0.0):.4f}")
        print(f"  ROC-AUC:   {val_metrics['roc_auc']:.4f}")

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        print(f"LR: {current_lr:.2e}")

        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'best_val_f1': best_val_f1,
            'channel_means': channel_means,
            'channel_stds': channel_stds,
            'reinhard_stats': reinhard_stats if reinhard_norm else None,
            'model_config': MODEL_CONFIG.copy(),
            'threshold': threshold,
        }

        torch.save(checkpoint, output_dir / f'{run_name}_last.pth')

        if epoch == 0 or val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            checkpoint['best_val_f1'] = best_val_f1
            torch.save(checkpoint, output_dir / f'{run_name}_best.pth')
            print(f"✓ Best model saved! (Val F1: {val_metrics['f1']:.4f})")

        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('loss/val', val_loss, epoch)
        writer.add_scalar('metric/val_accuracy', val_metrics['accuracy'], epoch)
        writer.add_scalar('metric/val_precision', val_metrics['precision'], epoch)
        writer.add_scalar('metric/val_recall', val_metrics['recall'], epoch)
        writer.add_scalar('metric/val_f1', val_metrics['f1'], epoch)
        writer.add_scalar('metric/val_dice_metric', val_metrics.get('dice_metric', 0.0), epoch)
        writer.add_scalar('metric/val_mean_iou', val_metrics.get('mean_iou', 0.0), epoch)
        writer.add_scalar('metric/val_roc_auc', val_metrics['roc_auc'], epoch)
        writer.add_scalar('lr', current_lr, epoch)

        train_history.append({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_metrics': val_metrics,
        })

    writer.close()

    print(f"\n{'='*60}")
    print("Final evaluation on test set...")
    print(f"{'='*60}")

    best_ckpt = torch.load(output_dir / f'{run_name}_best.pth', map_location=device)
    model.load_state_dict(best_ckpt['model_state_dict'])

    test_loss, test_metrics = eval_epoch(model, test_loader, criterion, device, split_name='test')
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Metrics:")
    print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
    print(f"  Precision: {test_metrics['precision']:.4f}")
    print(f"  Recall:    {test_metrics['recall']:.4f}")
    print(f"  F1:        {test_metrics['f1']:.4f}")
    print(f"  Dice:      {test_metrics.get('dice_metric', 0.0):.4f}")
    print(f"  mIoU:      {test_metrics.get('mean_iou', 0.0):.4f}")
    print(f"  ROC-AUC:   {test_metrics['roc_auc']:.4f}")

    report = {
        'run_name': run_name,
        'model_config': MODEL_CONFIG.copy(),
        'training_config': {
            'epochs': epochs,
            'batch_size': batch_size,
            'learning_rate': lr,
            'warmup_epochs': warmup_epochs,
            'use_amp': use_amp,
        },
        'normalization_params': {
            'reinhard_stats': reinhard_stats if reinhard_norm else None,
            'channel_means': channel_means,
            'channel_stds': channel_stds,
            'threshold': threshold,
        },
        'final_metrics': {
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_metrics': val_metrics,
            'test_loss': test_loss,
            'test_metrics': test_metrics,
        },
        'training_history': train_history,
    }

    with open(output_dir / f'{run_name}_report.json', 'w') as f:
        json.dump(report, f, indent=2)

    print(f"\n✓ Training complete!")
    print(f"Checkpoints saved to: {output_dir}")
    print(f"TensorBoard logs: tensorboard --logdir {log_dir}")
    
    return model, report


## Part 6: Usage Instructions

### Running Training

Uncomment and run the cell below to start training:

In [ ]:
if Path('Salidas/Tiles_UNet').exists():
    model, report = train_unet(
        epochs=50,
        batch_size=8,          # Optimized for T4 15.93 GB VRAM with AMP
        lr=1e-4,
        images_dir='Salidas/Tiles_UNet',
        output_dir='checkpoints',
        num_workers=0,      # Auto-calculate based on 28 GB RAM
        seed=42,
        warmup_epochs=5,
        accum_steps=8,
        use_amp=True,          # AMP (FP16) for T4 Tensor Cores — 2–3× speedup
        use_gradient_checkpointing=True,  # Save memory at cost of some speed (good for deeper models or larger batches)
    )
else:
    print("Run the preprocessing pipeline first:")
    print("  python tiling_unet.py")

Run: unet_binary_20260510_060404
Device: cuda
AMP (FP16): True
Output dir: checkpoints

--- Data Loading Pipeline ---
Step 1: Computing grouped split by biopsia...
  Train biopsies: 49
  Val biopsies: 11
  Test biopsies: 11

Step 2: Collecting train image paths...
  Train tiles: 10712

Step 3: Computing Reinhard stain normalization template...
  Template stats computed from ~200 samples
    L: mean=130.5, std=37.4
    a: mean=137.1, std=11.0
    b: mean=125.8, std=14.0

Step 4: Computing per-channel Z-score normalization stats...
  Channel means (RGB): ['0.5113', '0.4304', '0.4786']
  Channel stds (RGB):  ['0.1564', '0.2096', '0.0772']

Step 5: Creating augmentation pipelines...
  Using specified DataLoader workers: 0

Step 5b: Creating dataloaders with pre-computed splits and WeightedRandomSampler...


c:\Users\proyecto_final\Documents\Proyecto_Final_Glomerulos\env\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Computing duplicate-aware sampling weights for WeightedRandomSampler...
  Positive tiles (contain glomerulus): 2,677
  Negative tiles (background only):    8,035
  Sampler mode: duplicate-aware
  Unique glomeruli in train metadata: 1,275
  Primary tile links:   1,275
  Secondary tile links: 2,092
  Weight mass: pos=1.000, neg=1.000
  WeightedRandomSampler: 5,354 samples/epoch (~1:1 pos:neg ratio)
  Train: 10712 tiles | Val: 2013 tiles | Test: 2471 tiles

--- Mask Audit Report (pre-training) ---
Expected raw mask values: not enforced
Almost-full tile threshold: positive_area_ratio >= 0.80

[TRAIN]
  Tiles: 10,712 | positive: 2,677 | empty: 8,035 | almost full: 15
  Pixels: positive=476,958,341 negative=10,755,387,771 positive_ratio=0.042463
  Positive area ratio percentiles (all tiles): p0=0.000000, p1=0.000000, p5=0.000000, p25=0.000000, p50=0.000000, p75=0.000000, p95=0.285303, p99=0.586221, p100=0.997580
  Positive area ratio percentiles (positive tiles only): p0=0.000001, p1=0.00015

C:\Users\proyecto_final\AppData\Local\Temp\ipykernel_12268\1813271890.py:284: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (use_amp and device.type == 'cuda') else None


Model parameters: 24,436,369
AMP (FP16) enabled — GradScaler initialized

Starting training...

Epoch [1/2]


...

  Epoch 2 train done: avg_loss=0.1880 | avg_dice=0.293 | elapsed=1718.2s


  [val]:   0%|          | 0/2013 [00:00<?, ?batch/s]C:\Users\proyecto_final\AppData\Local\Temp\ipykernel_12268\382354600.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Val Loss: 0.1307
Val Metrics:
  Accuracy:  0.9471
  Precision: 0.3377
  Recall:    0.1149
  F1:        0.1715
  Dice:      0.1715
  mIoU:      0.0938
  ROC-AUC:   0.8715
LR: 1.00e-06

Final evaluation on test set...


  [test]:   0%|          | 0/2471 [00:00<?, ?batch/s]C:\Users\proyecto_final\AppData\Local\Temp\ipykernel_12268\382354600.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
                                                                             

Test Loss: 0.0835
Test Metrics:
  Accuracy:  0.9710
  Precision: 0.6511
  Recall:    0.6570
  F1:        0.6540
  Dice:      0.6540
  mIoU:      0.4859
  ROC-AUC:   0.9797

✓ Training complete!
Checkpoints saved to: checkpoints
TensorBoard logs: tensorboard --logdir checkpoints\unet_binary_20260510_060404


# Training Curves — Loss & Metrics Across Epochs

After training, this visualization shows how the model's performance improves over epochs.
We plot training vs validation metrics to detect overfitting and identify the best epoch.

In [ ]:
def visualize_training_report(report_path):
    """
    Load training report and visualize loss and metrics curves.
    
    Args:
        report_path: Path to report.json saved from train_unet
    """
    import json
    import matplotlib.pyplot as plt
    
    try:
        with open(report_path) as f:
            report = json.load(f)
    except FileNotFoundError:
        print(f"Report not found: {report_path}")
        return
    
    # Extract data
    train_loss = report.get('train_loss', [])
    val_loss = report.get('val_loss', [])
    val_metrics = report.get('val_metrics', {})
    best_epoch = report.get('best_epoch', 0)
    
    epochs = range(1, len(train_loss) + 1)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Loss
    ax = axes[0, 0]
    ax.plot(epochs, train_loss, 'o-', label='Train Loss', alpha=0.7, linewidth=2)
    ax.plot(epochs, val_loss, 's-', label='Val Loss', alpha=0.7, linewidth=2)
    ax.axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best Epoch ({best_epoch})')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Loss', fontsize=11)
    ax.set_title('Loss Curves', fontweight='bold', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: F1 Score
    ax = axes[0, 1]
    val_f1 = val_metrics.get('f1', [])
    if val_f1:
        ax.plot(epochs, val_f1, 'o-', color='orange', label='Val F1', linewidth=2)
        ax.axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel('F1 Score', fontsize=11)
        ax.set_title('F1 Score', fontweight='bold', fontsize=12)
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1])
    
    # Plot 3: Precision vs Recall
    ax = axes[1, 0]
    val_precision = val_metrics.get('precision', [])
    val_recall = val_metrics.get('recall', [])
    if val_precision and val_recall:
        ax.plot(epochs, val_precision, 'o-', label='Precision', linewidth=2)
        ax.plot(epochs, val_recall, 's-', label='Recall', linewidth=2)
        ax.axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel('Score', fontsize=11)
        ax.set_title('Precision vs Recall', fontweight='bold', fontsize=12)
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1])
    
    # Plot 4: Dice
    ax = axes[1, 1]
    val_dice = val_metrics.get('dice_metric', [])
    if val_dice:
        ax.plot(epochs, val_dice, '^-', color='green', label='Val Dice', linewidth=2)
        ax.axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best Epoch')
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel('Dice Score', fontsize=11)
        ax.set_title('Dice Metric', fontweight='bold', fontsize=12)
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1])
    
    plt.suptitle(f"Training Report — {Path(report_path).parent.name}", 
                fontsize=13, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
# If training finished, visualize the report
import glob
report_files = sorted(glob.glob('checkpoints/*_report.json'))
if report_files:
    latest_report = report_files[-1]
    visualize_training_report(latest_report)
    print(f"Report: {latest_report}")

In [18]:
from scipy import ndimage
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from skimage.measure import label, regionprops
import numpy as np

def apply_nms_to_predictions(
    pred_masks: list,
    pred_probs: list,
    tile_coords: list,
    iou_threshold: float = 0.5,
) -> list:
    """Apply distance-transform and watershed to separate instances.
    Instead of bounding box NMS, we use topological separation.
    """
    all_instances = []

    for tile_idx, (bin_mask, prob_map, coords) in enumerate(
        zip(pred_masks, pred_probs, tile_coords)
    ):
        x0_tile, y0_tile, x1_tile, y1_tile = coords
        tile_h, tile_w = bin_mask.shape

        scale_x = (x1_tile - x0_tile) / tile_w
        scale_y = (y1_tile - y0_tile) / tile_h

        # Distance transform and watershed
        distance = ndimage.distance_transform_edt(bin_mask)
        coords_peaks = peak_local_max(distance, min_distance=10, labels=bin_mask)
        mask_peaks = np.zeros(distance.shape, dtype=bool)
        mask_peaks[tuple(coords_peaks.T)] = True
        markers, _ = ndimage.label(mask_peaks)
        labels = watershed(-distance, markers, mask=bin_mask)

        props = regionprops(labels, intensity_image=prob_map)

        for region in props:
            if region.area < 100:
                continue

            r0, c0, r1, c1 = region.bbox
            wsi_x0 = x0_tile + c0 * scale_x
            wsi_y0 = y0_tile + r0 * scale_y
            wsi_x1 = x0_tile + c1 * scale_x
            wsi_y1 = y0_tile + r1 * scale_y

            confidence = float(region.mean_intensity)

            all_instances.append({
                'bbox': (wsi_x0, wsi_y0, wsi_x1, wsi_y1),
                'confidence': confidence,
            })

    # Return all detected instances (further merging across tiles can be done if needed)
    return [(d['bbox'], d['confidence']) for d in all_instances]


## Part 7: Troubleshooting and System Requirements

### Troubleshooting

**"No images found in Salidas/Estandarizados"**
- Run the preprocessing pipeline first (tiling -> normalization -> standardization).

**"CUDA out of memory"**
- Reduce `batch_size` (try 2 or 1)
- Increase `--ram-fraction` in preprocessing scripts
- Use CPU training (slower but uses less VRAM)

**"Expected masks directory"**
- Ensure masks were copied by normalizacion.py and estandarizacion.py to `Salidas/Estandarizados/*/masks/`
- Verify mask naming convention: `{tile_name}_mask.png`

**"No valid image-mask pairs found"**
- Check that image and mask counts match
- Verify paths: images in `*/images/` and masks in `*/masks/`

### System Requirements

- **Python**: 3.10+
- **PyTorch**: 2.0+ (with CUDA if GPU available)
- **RAM**: 16GB minimum, 32GB+ recommended
- **GPU**: Optional but recommended (4GB VRAM minimum)
- **Dependencies**: See requirements.txt (numpy, torch, torchvision, opencv, scikit-learn, albumentations, tensorboard)

### References

- **U-Net**: Ronneberger et al., "U-Net: Convolutional Networks for Biomedical Image Segmentation" (MICCAI 2015)
- **Dice Loss**: Sørensen–Dice coefficient for segmentation evaluation

In [19]:
def visualize_predictions(
    model,
    image_path,
    mask_path=None,
    device='cpu',
    threshold=0.5,
    reinhard_norm=None,
    channel_means=None,
    channel_stds=None,
):
    """Visualize model predictions on a test tile.
    
    Args:
        model: smp.Unet model (1-channel binary output)
        image_path: Path to input RGB image
        mask_path: Path to ground truth mask (optional)
        device: Torch device
        threshold: Sigmoid threshold for binary classification (default 0.5)
        reinhard_norm: Optional Reinhard normalizer used during training
        channel_means: Optional RGB channel means used during training
        channel_stds: Optional RGB channel stds used during training
    """
    from PIL import Image
    import matplotlib.pyplot as plt
    
    # Load image for display and model preprocessing
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img, dtype=np.uint8)

    # Match GlomeruliDataset preprocessing: RGB -> BGR -> Reinhard -> RGB/255 -> Z-score
    img_bgr = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)
    if reinhard_norm is not None:
        img_bgr = reinhard_norm(img_bgr)

    img_preprocessed = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    if channel_means is not None and channel_stds is not None:
        for c in range(3):
            img_preprocessed[..., c] = (
                img_preprocessed[..., c] - channel_means[c]
            ) / (channel_stds[c] + 1e-6)
    
    img_tensor = torch.from_numpy(np.transpose(img_preprocessed, (2, 0, 1))).float().unsqueeze(0)
    img_tensor = img_tensor.to(device)
    
    # Model forward pass
    model.eval()
    with torch.no_grad():
        logits = model(img_tensor)  # [1, 1, H, W]
        probs = torch.sigmoid(logits).squeeze(0).squeeze(0).cpu().numpy()  # [H, W]
    
    pred_mask = (probs > threshold).astype(np.uint8) * 255
    
    # Visualization
    n_cols = 3 if mask_path else 2
    fig, axes = plt.subplots(1, n_cols, figsize=(15, 5))
    
    axes[0].imshow(img_array)
    axes[0].set_title('Input Image')
    axes[0].axis('off')
    
    axes[1].imshow(pred_mask, cmap='gray')
    axes[1].set_title(f'Prediction (threshold={threshold})')
    axes[1].axis('off')
    
    if mask_path and Path(mask_path).exists():
        gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        axes[2].imshow(gt_mask, cmap='gray')
        axes[2].set_title('Ground Truth')
        axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return pred_mask


# Detailed Prediction Evaluation — Ground Truth vs Predictions

This section shows the model's predictions compared to ground truth with pixel-level error analysis.
Each column shows a different view: original image, ground truth mask, predicted mask, overlay, and error map (TP/FP/FN).

In [ ]:
def visualize_prediction_comparison(model, image_path, mask_path, device, threshold=0.5, 
                                  reinhard_norm=None, channel_means=None, channel_stds=None):
    """
    Detailed prediction comparison: GT vs Pred with error analysis.
    
    Args:
        model: Trained U-Net model
        image_path: Path to input tile image
        mask_path: Path to ground truth mask
        device: Torch device
        threshold: Probability threshold for binarization
        reinhard_norm: ReinhardNormalize instance
        channel_means: Channel means for Z-score
        channel_stds: Channel stds for Z-score
    """
    import matplotlib.pyplot as plt
    import cv2
    import torch
    
    model.eval()
    
    # Load and preprocess image
    img_rgb = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    mask_gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask_gt_binary = (mask_gt > 0).astype(np.uint8)
    
    # Preprocess: Reinhard, Z-score
    img_prep = img_rgb.copy()
    if reinhard_norm:
        img_prep = reinhard_norm(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
        img_prep = cv2.cvtColor(img_prep, cv2.COLOR_BGR2RGB)
    
    # Normalize
    img_tensor = img_prep.astype(np.float32) / 255.0
    if channel_means and channel_stds:
        for c in range(3):
            img_tensor[..., c] = (img_tensor[..., c] - channel_means[c]) / (channel_stds[c] + 1e-8)
    
    # Transpose and add batch
    img_tensor = torch.from_numpy(np.transpose(img_tensor, (2, 0, 1))).float().unsqueeze(0)
    img_tensor = img_tensor.to(device)
    
    # Predict
    with torch.no_grad():
        logits = model(img_tensor)
        pred_prob = torch.sigmoid(logits)
    
    pred_prob_np = pred_prob.squeeze().cpu().numpy()
    pred_binary = (pred_prob_np > threshold).astype(np.uint8)
    
    # Compute metrics for this tile
    tp = np.sum((mask_gt_binary == 1) & (pred_binary == 1))
    fp = np.sum((mask_gt_binary == 0) & (pred_binary == 1))
    fn = np.sum((mask_gt_binary == 1) & (pred_binary == 0))
    tn = np.sum((mask_gt_binary == 0) & (pred_binary == 0))
    
    dice = 2 * tp / (2 * tp + fp + fn + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    
    # Create visualizations
    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    
    # 1. Original image
    axes[0].imshow(img_rgb)
    axes[0].set_title("1. Original Image", fontweight='bold', fontsize=11)
    axes[0].axis('off')
    
    # 2. Ground truth
    axes[1].imshow(mask_gt_binary, cmap='RdYlGn', vmin=0, vmax=1)
    axes[1].set_title("2. Ground Truth\n(white=glomerulus)", fontweight='bold', fontsize=11)
    axes[1].axis('off')
    
    # 3. Prediction
    axes[2].imshow(pred_binary, cmap='Blues', vmin=0, vmax=1)
    axes[2].set_title("3. Predicted Mask\n(threshold=0.5)", fontweight='bold', fontsize=11)
    axes[2].axis('off')
    
    # 4. Overlay (GT green + Pred red)
    overlay = img_rgb.copy().astype(float)
    overlay[mask_gt_binary == 1] = [0, 255, 0]  # GT in green
    overlay[pred_binary == 1] = [255, 0, 0]     # Pred in red
    overlay = overlay.astype(np.uint8)
    alpha = 0.4
    blended = cv2.addWeighted(img_rgb, 1-alpha, overlay, alpha, 0)
    axes[3].imshow(blended)
    axes[3].set_title("4. Overlay\n(GT=green, Pred=red)", fontweight='bold', fontsize=11)
    axes[3].axis('off')
    
    # 5. Error map: TP/FP/FN
    error_map = np.zeros((*mask_gt_binary.shape, 3), dtype=np.uint8)
    error_map[(mask_gt_binary == 1) & (pred_binary == 1)] = [0, 255, 0]    # TP green
    error_map[(mask_gt_binary == 0) & (pred_binary == 1)] = [255, 0, 0]    # FP red
    error_map[(mask_gt_binary == 1) & (pred_binary == 0)] = [255, 255, 0]  # FN yellow
    
    axes[4].imshow(error_map)
    axes[4].set_title("5. Error Map\nTP=green, FP=red, FN=yellow", fontweight='bold', fontsize=11)
    axes[4].axis('off')
    
    # Add metrics as text below
    metrics_text = f"Dice={dice:.3f} | IoU={iou:.3f} | Prec={precision:.3f} | Recall={recall:.3f}"
    fig.text(0.5, 0.02, metrics_text, ha='center', fontsize=11, 
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle(f"Prediction Analysis — {Path(image_path).name}", 
                fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
if Path('Salidas/Tiles_UNet').exists():
    # Load model and visualize predictions on test samples
    checkpoint_dir = Path('checkpoints')
    best_checkpoints = sorted(checkpoint_dir.glob('*_best.pth'))
    
    if best_checkpoints:
        best_ckpt = best_checkpoints[-1]
        
        # Load model
        model_eval = create_model(**MODEL_CONFIG).to(device)
        checkpoint = torch.load(best_ckpt, map_location=device)
        # Extract preprocessing parameters from checkpoint
        channel_means = checkpoint.get('channel_means')
        channel_stds = checkpoint.get('channel_stds')
        reinhard_stats = checkpoint.get('reinhard_stats')
        model_eval.load_state_dict(checkpoint['model_state_dict'])
        
        # Get test samples
        reinhard_norm = ReinhardNormalize(reinhard_stats) if reinhard_stats else None
        ds_test = GlomeruliDataset('Salidas/Tiles_UNet', split='test',
                                 reinhard_norm=reinhard_norm, channel_means=channel_means, channel_stds=channel_stds)
        positive_test_idx = [i for i, flag in enumerate(ds_test.get_positive_flags()) if flag]
        
        if positive_test_idx:
            import random
            random.seed(42)
            sample_indices = random.sample(positive_test_idx, min(50, len(positive_test_idx)))
            
            for sample_idx in sample_indices:
                img_path = ds_test.image_paths[sample_idx]
                mask_path = ds_test._get_mask_path(img_path)
                visualize_prediction_comparison(model_eval, img_path, mask_path, device)
        
        print(f"Loaded checkpoint: {best_ckpt.name}")
    else:
        print("No trained model checkpoint found")

## Part 8: Qualitative Evaluation — Predictions vs Ground Truth

Visual inspection of model predictions on the test set is critical in medical image segmentation. Quantitative metrics like F1 and accuracy can mask clinically relevant errors such as:
- Imprecise borders around glomeruli (false positives/negatives at edges)
- False positives in glomerulus-like structures (cracks, debris)
- Missed glomeruli in hard-to-segment regions

The cells below visualize predictions alongside ground truth for qualitative assessment.